In [41]:
import sys
sys.path.insert(0, '../')    
from pathlib import Path
from config.config import config


In [43]:
import pandas as pd
import json

# Load both JSONs

# to get the with_finetuned.json - run: python -m scripts.run_ocr_batch --use_pretrained True --name "with_finetu"

# to get the without_finetuned.json - run: python -m scripts.run_ocr_batch --name "without_finetuned"
with open(config.processed_data_dir /"with_finetuned.json") as f:
    both = json.load(f)

with open(config.processed_data_dir /"without_finetuned.json") as f:
    pretrained = json.load(f)

# Convert to Series first (dict -> Series is a one-liner), then combine into a DataFrame
df = pd.DataFrame({
    "finetuned_pred": pd.Series(both),
    "pretrained_pred": pd.Series(pretrained),
})

df.index.name = "filename"
df = df.reset_index()

# Flag mismatches
df["match"] = df["finetuned_pred"] == df["pretrained_pred"]

print(f"Total: {len(df)}")
print(f"Matches: {df['match'].sum()}")
print(f"Mismatches: {(~df['match']).sum()}")

# Show the disagreements
mismatches = df[~df["match"]]
mismatches

Total: 951
Matches: 548
Mismatches: 403


,filename,finetuned_pred,pretrained_pred,match
9,License (101).png,MH02JP1724,M223311Y2,False
12,License (1015).png,GJ1HM2973,GJIHN19973,False
13,License (1017).png,MH12DM4009,M212DM4009,False
15,License (1021).png,MH04DN4957,NHOQDN49,False
23,License (1035).png,KL07BU7236,KL078U,False
...,...,...,...,...
941,License (979).png,DL2CAN4832,DL2CNN833,False
942,License (981).png,MH02CL7324,MHO2CL7324,False
944,License (985).png,AP29AJ8241,A929AJ8241,False
948,License (993).png,TN02AQ8660,TNO2AQ8660,False


In [44]:

crops_dir = config.processed_data_dir / "Indian_LPR_deduped"

for _, row in mismatches.iterrows():
    src = crops_dir / row["filename"]
    if not src.exists():
        print(f"MISSING: {src}")
        continue
    print(f"{row['filename']}: finetuned={row['finetuned_pred']} | pretrained={row['pretrained_pred']}")


out_dir = config.processed_data_dir / "LPR_mismatched"
out_dir.mkdir(exist_ok=True)
import shutil
for _, row in mismatches.iterrows():
    src = crops_dir / row["filename"]
    if src.exists():
        shutil.copy(src, out_dir / row["filename"])

mismatches.to_csv(out_dir / "labels.csv", index=False)
print(f"\nCopied {len(mismatches)} images + labels.csv to {out_dir}")

License (101).png: finetuned=MH02JP1724 | pretrained=M223311Y2
License (1015).png: finetuned=GJ1HM2973 | pretrained=GJIHN19973
License (1017).png: finetuned=MH12DM4009 | pretrained=M212DM4009
License (1021).png: finetuned=MH04DN4957 | pretrained=NHOQDN49
License (1035).png: finetuned=KL07BU7236 | pretrained=KL078U
License (1043).png: finetuned=AP23R1651 | pretrained=AP21656
License (1045).png: finetuned=MH04CT2218 | pretrained=MHOCC2
License (1047).png: finetuned=MH04CT2218 | pretrained=NHAAM
License (1049).png: finetuned=MH04CT2218 | pretrained=NUAAM4
License (105).png: finetuned=MH12EG5878 | pretrained=MM2GG58878
License (1051).png: finetuned=GJ1HN8098 | pretrained=GJ1HN89988
License (1061).png: finetuned=WB24R1011 | pretrained=WB24R101
License (107).png: finetuned=MH12EG5878 | pretrained=MH12EG587
License (1073).png: finetuned=TN07AP7859 | pretrained=TNOTA1
License (1083).png: finetuned=TN20BT7899 | pretrained=TN2OBT7899
License (1091).png: finetuned=MH02BM1873 | pretrained=MHO2BM18

In [49]:
data_dir = config.processed_data_dir/ "ocr_labels" / "Indian_LPR_deduped"
df = pd.read_csv(data_dir / "ocr_labels.csv")

In [50]:
df = df[517:]

In [52]:
df.to_csv(data_dir/'unseen.csv', index=False)


Evaluate the base pretrained OCR model vs. the fine-tuned OCR model on a held-out,
human-verified ground-truth set (unseen.csv), and surface exactly where they disagree.

unseen.csv is expected at config.processed_data_dir / "ocr_labels" / "unseen.csv"
with columns: filename, predicted_text, corrected_text, was_correct
(was_correct/predicted_text refer to whichever model originally drafted the labels
during the labeling pass — they are NOT used for accuracy here. corrected_text,
human-verified, is the only ground truth this script trusts.)

Cropped plate images referenced by `filename` are expected under:
    config.processed_data_dir / "Indian_LPR_deduped"


In [74]:

import csv
from pathlib import Path

from PIL import Image

from config.config import config
from src.ocr import load_ocr_model, load_finetuned_ocr_model

CSV_PATH = config.processed_data_dir / "ocr_labels" / "Indian_LPR_deduped" / "unseen.csv"
IMAGES_DIR = config.processed_data_dir / "Indian_LPR_deduped"


def normalize(text: str) -> str:
    """Uppercase + strip whitespace before comparing, so formatting quirks
    (stray spaces, lowercase) don't get counted as OCR errors."""
    return (text or "").strip().upper()


def load_ground_truth(csv_path: Path) -> list[dict]:
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def run_model_as_single_line(model, cropped_plate: Image.Image) -> str | None:
    """
    Runs a single OCR model directly, bypassing recognize_plate()'s two-line
    routing — we want to compare the base and fine-tuned models head-to-head
    on the exact same input, not have the two-line heuristic silently swap in
    a different model for either of them.
    """
    import numpy as np
    result = model.run(np.array(cropped_plate.convert("RGB")))
    if not result:
        return None
    return result[0].plate


def evaluate():
    if not CSV_PATH.exists():
        raise FileNotFoundError(f"Ground-truth CSV not found at {CSV_PATH}")
    if not IMAGES_DIR.is_dir():
        raise NotADirectoryError(f"Image folder not found at {IMAGES_DIR}")

    rows = load_ground_truth(CSV_PATH)
    print(f"Loaded {len(rows)} ground-truth rows from {CSV_PATH}")

    base_model = load_ocr_model()
    finetuned_model = load_finetuned_ocr_model()

    base_correct = 0
    finetuned_correct = 0
    disagreements = []
    missing_images = []

    for row in rows:
        filename = row["filename"]
        gt_text = normalize(row["corrected_text"])

        img_path = IMAGES_DIR / filename
        if not img_path.exists():
            missing_images.append(filename)
            continue

        img = Image.open(img_path)

        base_pred = normalize(run_model_as_single_line(base_model, img))
        finetuned_pred = normalize(run_model_as_single_line(finetuned_model, img))

        base_is_correct = base_pred == gt_text
        finetuned_is_correct = finetuned_pred == gt_text

        if base_is_correct:
            base_correct += 1
        if finetuned_is_correct:
            finetuned_correct += 1

        if base_pred != finetuned_pred:
            disagreements.append({
                "filename": filename,
                "ground_truth": gt_text,
                "base_pred": base_pred,
                "finetuned_pred": finetuned_pred,
                "base_correct": base_is_correct,
                "finetuned_correct": finetuned_is_correct,
            })

    total = len(rows) - len(missing_images)
    if missing_images:
        print(f"\nWARNING: {len(missing_images)} filenames from the CSV were not found in {IMAGES_DIR}:")
        for name in missing_images[:10]:
            print(f"  - {name}")
        if len(missing_images) > 10:
            print(f"  ... and {len(missing_images) - 10} more")

    print(f"\nEvaluated on {total} images")
    print(f"Base pretrained accuracy:    {base_correct}/{total} = {base_correct/total:.2%}")
    print(f"Fine-tuned accuracy:         {finetuned_correct}/{total} = {finetuned_correct/total:.2%}")

    print(f"\nDisagreements between models: {len(disagreements)}/{total} = {len(disagreements)/total:.2%}")

    finetuned_wins = sum(1 for d in disagreements if d["finetuned_correct"] and not d["base_correct"])
    base_wins = sum(1 for d in disagreements if d["base_correct"] and not d["finetuned_correct"])
    both_wrong = sum(1 for d in disagreements if not d["base_correct"] and not d["finetuned_correct"])

    print(f"  Fine-tuned correct, base wrong: {finetuned_wins}")
    print(f"  Base correct, fine-tuned wrong: {base_wins}")
    print(f"  Both wrong (different ways):    {both_wrong}")

    # Save the full disagreement breakdown so you can visually inspect the interesting cases
    output_path = config.processed_data_dir / "ocr_labels" / "model_disagreements.csv"
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "filename", "ground_truth", "base_pred", "finetuned_pred",
            "base_correct", "finetuned_correct"
        ])
        writer.writeheader()
        writer.writerows(disagreements)
    print(f"\nFull disagreement list saved to {output_path}")


In [75]:
evaluate()

Loaded 134 ground-truth rows from C:\SHASWAT\Projects\Vehicle-Monitoring-System\data\processed\ocr_labels\Indian_LPR_deduped\unseen.csv
*************** EP Error ***************
EP Error N:\_work\1\s\onnxruntime\python\onnxruntime_pybind_state.cc:534 onnxruntime::python::RegisterTensorRTPluginsAsCustomOps Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
Falling back to ['CUDAExecutionProvider', 'CPUExecutionProvider'] and retrying.
****************************************
*************** EP Error ***************
EP Error N:\_work\1\s\onnxruntime\python\onnxruntime_pybind_state.cc:534 onnxruntime::python::RegisterTensorRTPluginsAsCustomOps Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is suppo


Evaluated on 134 images
Base pretrained accuracy:    71/134 = 52.99%
Fine-tuned accuracy:         112/134 = 83.58%

Disagreements between models: 61/134 = 45.52%
  Fine-tuned correct, base wrong: 42
  Base correct, fine-tuned wrong: 1
  Both wrong (different ways):    18

Full disagreement list saved to C:\SHASWAT\Projects\Vehicle-Monitoring-System\data\processed\ocr_labels\model_disagreements.csv


In [76]:
model_disagreement = Path("C:\SHASWAT\Projects\Vehicle-Monitoring-System\data\processed\ocr_labels\model_disagreements.csv")

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\sys_a\AppData\Local\Temp\ipykernel_1920\1753599136.py:1: SyntaxWarning: invalid escape sequence '\S'
  model_disagreement = Path("C:\SHASWAT\Projects\Vehicle-Monitoring-System\data\processed\ocr_labels\model_disagreements.csv")


In [77]:
df = pd.read_csv(model_disagreement)

# Exclude unreadable/partial ground truth from the full 134 (you'll need the full labeled set, not just disagreements)
# For this disagreements-only file, fix the known typos:
fixes = {
    "License (325).png": "GJ1HR9202",
    "License (383).png": "GJ18AB8628",
    "License (441).png": "MH02BY7659",
}
for fname, correct in fixes.items():
    df.loc[df["filename"] == fname, "ground_truth"] = correct

df["base_correct"] = df["base_pred"] == df["ground_truth"]
df["finetuned_correct"] = df["finetuned_pred"] == df["ground_truth"]
print(df[["base_correct","finetuned_correct"]].value_counts())

base_correct  finetuned_correct
False         True                 42
              False                18
True          False                 1
Name: count, dtype: int64


In [62]:
from src.detection import run_inference_and_crop

run_inference_and_crop(model_path=config.yolo100ep_best_weights,
                       source_dir=config.raw_data_dir / "two_line_plates",
                       annotated_dir=config.processed_data_dir / 'two_line_plates' / 'annotated',
                       crops_dir=config.processed_data_dir / 'two_line_plates' / 'crops')

Running inference on 30 images
Annotated images saved to C:\SHASWAT\Projects\Vehicle-Monitoring-System\data\processed\two_line_plates\annotated
Cropped plates saved to C:\SHASWAT\Projects\Vehicle-Monitoring-System\data\processed\two_line_plates\crops


In [78]:
df

,filename,ground_truth,base_pred,finetuned_pred,base_correct,finetuned_correct
0,License (223).png,MH04EH5222,MH04EH5EE,MH04EH522Z,False,False
1,License (243).png,TNO1AL2508,TNO1AL2508,TN01AL2508,True,False
2,License (25).png,WB02AA3572,WB2AA355,WB02AA3572,False,True
3,License (253).png,KA04MA3873,KAOMA3873,KA04MA3873,False,True
4,License (257).png,KA03MN5959,KAO3MN5959,KA03MN5959,False,True
...,...,...,...,...,...,...
56,License (443).png,KA05MG5789,5606578,KAO5MG5789,False,False
57,License (445).png,MH14BC8780,MH14BC878,MH14BC8780,False,True
58,License (45).png,MH04FA6281,MIAAAA,MH04F16281,False,False
59,License (451).png,MH01AC2577,MHOIAC2577,MH0IAC2577,False,False


In [88]:
files = df[(~df['base_correct']) & (~df['finetuned_correct'])]
files = files['filename'].tolist()

In [89]:
from pathlib import Path
import shutil

# Source and destination folders
source_dir = config.processed_data_dir / "Indian_LPR_deduped"
destination_dir = config.processed_data_dir / 'wrong_images'

# Create destination folder if it doesn't exist
destination_dir.mkdir(parents=True, exist_ok=True)

# List of filenames to copy

# Copy files
for filename in files:
    src = source_dir / filename
    dst = destination_dir / filename

    if src.exists():
        shutil.copy2(src, dst)
        print(f"Copied: {filename}")
    else:
        print(f"File not found: {filename}")

print("Done!")

Copied: License (223).png
Copied: License (259).png
Copied: License (27).png
Copied: License (287).png
Copied: License (307).png
Copied: License (321).png
Copied: License (341).png
Copied: License (343).png
Copied: License (375).png
Copied: License (39).png
Copied: License (413).png
Copied: License (423).png
Copied: License (427).png
Copied: License (431).png
Copied: License (433).png
Copied: License (443).png
Copied: License (45).png
Copied: License (451).png
Done!
